# Adversarial Robustness Evaluation Framework for NIDS

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- INSTALL REQUIRED PACKAGES ---
!pip install -q pandas scikit-learn imbalanced-learn torch torchvision adversarial-robustness-toolbox>=1.17

In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime
import os

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, CarliniL2Method
from art.defences.trainer import AdversarialTrainer

# ============================================================================
# CONFIGURATION SECTION - Modify parameters here
# ============================================================================

# Dataset configuration
DATASET_PATH = 'drive/My Drive/CSE_CIC_IDS2018.csv'
DATASET_NAME = 'CSECICIDS2018'

#DATASET_PATH = 'drive/My Drive/wustl-ehms-2020_with_attacks_categories.csv'
#DATASET_NAME = 'WUSTLEHMS2020'

#DATASET_PATH = 'drive/My Drive/ton_iot_dataset.csv'
#DATASET_NAME = 'TONIOT'


# General settings
RANDOM_SEED = 42
TEST_SIZE = 0.2
FEATURE_SELECTION = 'auto'  # 'auto' or specify number (e.g., 20)

# Attack configurations
ATTACK_CONFIGS = {
    'FGSM': {
        'epsilon_values': [0.15, 0.30, 0.50, 0.75]
    },
    'PGD': {
        'epsilon_values': [0.15, 0.30, 0.50, 0.75],
        'max_iter_values': [10, 16, 25]
    },
    'CW': {
        'confidence_values': [0.0, 0.5],
        'max_iter_values': [5, 10]
    }
}

# Defense configurations
DEFENSE_CONFIGS = {
    'adversarial_training': {
        'ratio_values': [0.3, 0.4, 0.5],
        'nb_epochs': 8,
        'attack_types': ['FGSM', 'PGD']  # Which attacks to use for training
    },
    'randomized_smoothing': {
        'sigma_values': [0.1, 0.25, 0.5, 1.0],
        'n0': 100,
        'n': 500,
        'alpha': 0.001,
        'test_subset_size': 200
    }
}

# Model training configurations
MLP_CONFIG = {
    'hidden_layers': [32, 12],
    'nb_classes': 2,
    'lr': 0.001,
    'batch_size': 32,
    'nb_epochs': 12
}

ML_MODELS_CONFIG = {
    'RF': {'n_estimators': 50, 'random_state': RANDOM_SEED},
    'SVM': {'max_iter': 300, 'random_state': RANDOM_SEED},
    'LR': {'max_iter': 300, 'random_state': RANDOM_SEED}
}

# Output settings
OUTPUT_DIR = 'drive/My Drive/adversarial_results3'
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def create_output_directory():
    """Create output directory if it doesn't exist"""
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    print(f"Results will be saved to: {OUTPUT_DIR}")

def save_results_to_csv(results_df, filename):
    """Save results DataFrame to CSV file"""
    filepath = os.path.join(OUTPUT_DIR, f"{filename}_{DATASET_NAME}.csv")
    results_df.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")
    return filepath

class ResultsCollector:
    """Class to systematically collect and organize experiment results"""

    def __init__(self):
        self.base_results = []
        self.attack_results = []
        self.defense_results = []
        self.comparison_results = []

    def add_base_result(self, model_name, accuracy, precision, recall, f1_score, fpr, fnr):
        """Add baseline model performance"""
        self.base_results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1_score,
            'FPR': fpr,
            'FNR': fnr
        })

    def add_attack_result(self, model_name, attack_type, attack_params,
                         accuracy, precision, recall, f1_score, fpr, fnr, asr):
        """Add adversarial attack evaluation result"""
        result = {
            'Model': model_name,
            'Attack Type': attack_type,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1_score,
            'FPR': fpr,
            'FNR': fnr,
            'ASR': asr
        }
        result.update(attack_params)
        self.attack_results.append(result)

    def add_defense_result(self, model_name, defense_type, defense_params,
                          eval_type, eval_params, accuracy, precision, recall, f1_score, fpr, fnr, cert_accuracy=0, mean_certified_radius=0):
        """Add defense evaluation result"""
        result = {
            'Model': model_name,
            'Defense Type': defense_type,
            'Evaluation Type': eval_type,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1_score,
            'FPR': fpr,
            'FNR': fnr,
            'Certified Accuracy': cert_accuracy,
            'Mean Certified Radius': mean_certified_radius
        }
        result.update({f'Defense_{k}': v for k, v in defense_params.items()})
        result.update({f'Eval_{k}': v for k, v in eval_params.items()})
        self.defense_results.append(result)

    def get_dataframes(self):
        """Return all results as DataFrames"""
        return {
            'base_results': pd.DataFrame(self.base_results),
            'attack_results': pd.DataFrame(self.attack_results),
            'defense_results': pd.DataFrame(self.defense_results)
        }

    def save_all_results(self):
        """Save all results to CSV files"""
        dfs = self.get_dataframes()
        saved_files = []

        for name, df in dfs.items():
            if not df.empty:
                filepath = save_results_to_csv(df, name)
                saved_files.append(filepath)

        return saved_files

# ============================================================================
# DATA PREPROCESSING FUNCTIONS
# ============================================================================

def determine_optimal_features(X, y, max_features=50, cv_folds=5):
    """Determine optimal number of features using cross-validation"""
    rf = RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED)
    rf.fit(X, y)
    importances = rf.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_SEED)
    feature_range = range(5, min(max_features, len(sorted_idx)) + 1, 5)
    if len(feature_range) == 0:
        feature_range = range(5, len(sorted_idx) + 1, 5)

    best_score = -1
    best_n_features = min(20, len(sorted_idx))

    print("Finding optimal number of features...")
    for n_features in feature_range:
        selected_idx = sorted_idx[:n_features]
        X_selected = X.iloc[:, selected_idx]

        clf = RandomForestClassifier(n_estimators=20, random_state=RANDOM_SEED)
        cv_scores = cross_val_score(clf, X_selected, y, cv=cv, scoring='f1_macro')
        mean_score = np.mean(cv_scores)

        print(f"Features: {n_features}, Mean F1-Score: {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_n_features = n_features

    print(f"\nOptimal number of features: {best_n_features} (F1-Score: {best_score:.4f})")
    return best_n_features, sorted_idx

def preprocessing_and_feature_selection(df, label_col, select_n='auto',
                                       scaler=None, encoder=None):
    """Preprocess data and perform feature selection"""
    X, y = df.drop(columns=[label_col]), df[label_col]

    # Encode categorical features
    categ_cols = X.select_dtypes(include='object').columns
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

    for col in categ_cols:
        enc = encoder or LabelEncoder()
        X[col] = enc.fit_transform(X[col])

    scl = scaler or StandardScaler()
    X[numeric_cols] = scl.fit_transform(X[numeric_cols])
    X = pd.DataFrame(X, columns=X.columns)

    # Feature selection
    if select_n == 'auto':
        select_n, sorted_idx = determine_optimal_features(X, y)
        idx_top = sorted_idx[:select_n]
    else:
        rf_init = RandomForestClassifier(n_estimators=20, random_state=RANDOM_SEED)
        rf_init.fit(X, y)
        importances = rf_init.feature_importances_
        idx_top = np.argsort(importances)[-select_n:]

    X, cols_selected = X.iloc[:, idx_top], X.columns[idx_top]
    return X, y, scl, cols_selected

# ============================================================================
# MODEL DEFINITION
# ============================================================================

class Net(nn.Module):
    """MLP Neural Network"""
    def __init__(self, input_size, hidden_layers=[32, 12], num_classes=2):
        super().__init__()
        layers = []
        prev_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# ============================================================================
# EVALUATION FUNCTIONS
# ============================================================================

def evaluate_model(model, X, y, model_type='sklearn'):
    """Evaluate model and return metrics"""
    if model_type == 'pytorch':
        preds = model.predict(X.astype(np.float32))
        pred_labels = np.argmax(preds, axis=1)
    else:
        pred_labels = model.predict(X)

    accuracy = accuracy_score(y, pred_labels)
    precision = precision_score(y, pred_labels, zero_division=0)
    recall = recall_score(y, pred_labels, zero_division=0)
    f1 = f1_score(y, pred_labels, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y, pred_labels).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return accuracy, precision, recall, f1, fpr, fnr


def evaluate_adversarial_attack(model, X, adv_X, y, model_type='sklearn'):
    """Evaluate model and return metrics"""
    if model_type == 'pytorch':
        preds = model.predict(X.astype(np.float32))
        pred_labels = np.argmax(preds, axis=1)
        preds_adv = model.predict(adv_X.astype(np.float32))
        pred_labels_adv = np.argmax(preds_adv, axis=1)
    else:
        pred_labels = model.predict(X)
        pred_labels_adv = model.predict(adv_X)

    accuracy = accuracy_score(y, pred_labels_adv)
    precision = precision_score(y, pred_labels_adv, zero_division=0)
    recall = recall_score(y, pred_labels_adv, zero_division=0)
    f1 = f1_score(y, pred_labels_adv, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y, pred_labels_adv).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0


    # ASR - How many correctly classified samples become incorrect
    asr = np.mean((pred_labels == y) & (pred_labels_adv != y))


    return accuracy, precision, recall, f1, fpr, fnr, asr

# ============================================================================
# ATTACK GENERATION FUNCTIONS
# ============================================================================

def generate_attacks(classifier, X_test_np, attack_configs):
    """Generate adversarial examples for all configured attacks"""
    adversarial_examples = {}

    print("\n" + "="*60)
    print("GENERATING ADVERSARIAL EXAMPLES")
    print("="*60)

    # FGSM attacks
    if 'FGSM' in attack_configs:
        print("\nGenerating FGSM attacks...")
        for eps in attack_configs['FGSM']['epsilon_values']:
            attack_name = f'FGSM_eps{eps}'
            print(f"  - {attack_name}")
            fgsm = FastGradientMethod(estimator=classifier, eps=eps)
            adversarial_examples[attack_name] = {
                'data': fgsm.generate(X_test_np),
                'params': {'epsilon': eps},
                'attack': fgsm
            }

    # PGD attacks
    if 'PGD' in attack_configs:
        print("\nGenerating PGD attacks...")
        for eps in attack_configs['PGD']['epsilon_values']:
            for max_iter in attack_configs['PGD']['max_iter_values']:
                attack_name = f'PGD_eps{eps}_iter{max_iter}'
                print(f"  - {attack_name}")
                pgd = ProjectedGradientDescent(
                    estimator=classifier,
                    eps=eps,
                    max_iter=max_iter
                )
                adversarial_examples[attack_name] = {
                    'data': pgd.generate(X_test_np),
                    'params': {'epsilon': eps, 'max_iter': max_iter},
                    'attack': pgd
                }

    # CW attacks
    if 'CW' in attack_configs:
        print("\nGenerating C&W attacks...")
        for confidence in attack_configs['CW']['confidence_values']:
            for max_iter in attack_configs['CW']['max_iter_values']:
                attack_name = f'CW_conf{confidence}_iter{max_iter}'
                print(f"  - {attack_name}")
                cw = CarliniL2Method(
                    classifier,
                    confidence=confidence,
                    max_iter=max_iter
                )
                adversarial_examples[attack_name] = {
                    'data': cw.generate(X_test_np),
                    'params': {'confidence': confidence, 'max_iter': max_iter},
                    'attack': cw
                }

    print(f"\nGenerated {len(adversarial_examples)} attack variants")
    return adversarial_examples

# ============================================================================
# DEFENSE FUNCTIONS
# ============================================================================

def adversarial_training_defense(base_classifier, X_train, y_train, X_test, y_test,
                                attack, ratio, nb_epochs, adversarial_examples,
                                ml_models, results_collector):
    """Perform adversarial training defense"""

    attack_name = [k for k, v in adversarial_examples.items() if v['attack'] == attack][0]
    defense_params = {'ratio': ratio, 'nb_epochs': nb_epochs, 'attack': attack_name}

    print(f"\nAdversarial Training: {attack_name}, ratio={ratio}, epochs={nb_epochs}")

    # Train MLP with adversarial training
    adv_trainer = AdversarialTrainer(base_classifier, attacks=attack, ratio=ratio)
    adv_trainer.fit(
        X_train.values.astype(np.float32),
        y_train.values.astype(np.int64),
        nb_epochs=nb_epochs
    )
    mlp_defended = adv_trainer.get_classifier()

    # Evaluate MLP on clean data
    acc, prec, rec, f1, fpr, fnr = evaluate_model(
        mlp_defended,
        X_test.values.astype(np.float32),
        y_test.values,
        'pytorch'
    )
    results_collector.add_defense_result(
        'MLP', 'Adversarial_Training', defense_params,
        'Clean', {}, acc, prec, rec, f1, fpr, fnr, 0, 0
    )
    print(f"  MLP on clean: Acc={acc:.4f}, F1={f1:.4f}")

    # Evaluate MLP on adversarial examples
    for adv_name, adv_data in adversarial_examples.items():
        acc, prec, rec, f1, fpr, fnr = evaluate_model(
            mlp_defended,
            adv_data['data'],
            y_test.values,
            'pytorch'
        )
        results_collector.add_defense_result(
            'MLP', 'Adversarial_Training', defense_params,
            'Adversarial', {'attack': adv_name}, acc, prec, rec, f1, fpr, fnr, 0, 0
        )

    # Train traditional ML models with adversarial training
    X_train_adv = attack.generate(X_train.values.astype(np.float32))
    X_aug = np.vstack([X_train, X_train_adv])
    y_aug = np.concatenate([y_train, y_train])

    idx = np.random.permutation(len(y_aug))
    X_aug, y_aug = X_aug[idx], y_aug[idx]

    # Train and evaluate each ML model
    for model_name, model_class in ml_models.items():
        if model_name == 'RF':
            model = RandomForestClassifier(**ML_MODELS_CONFIG['RF'])
        elif model_name == 'SVM':
            model = LinearSVC(**ML_MODELS_CONFIG['SVM'])
        else:
            model = LogisticRegression(**ML_MODELS_CONFIG['LR'])

        model.fit(X_aug, y_aug)

        # Evaluate on clean data
        acc, prec, rec, f1, fpr, fnr = evaluate_model(model, X_test.values, y_test.values, 'sklearn')
        results_collector.add_defense_result(
            model_name, 'Adversarial_Training', defense_params,
            'Clean', {}, acc, prec, rec, f1, fpr, fnr, 0, 0
        )

        # Evaluate on adversarial examples
        for adv_name, adv_data in adversarial_examples.items():
            acc, prec, rec, f1, fpr, fnr = evaluate_model(
                model,
                adv_data['data'],
                y_test.values,
                'sklearn'
            )
            results_collector.add_defense_result(
                model_name, 'Adversarial_Training', defense_params,
                'Adversarial', {'attack': adv_name}, acc, prec, rec, f1, fpr, fnr, 0, 0
            )

def randomized_smoothing_defense(models, X_test, y_test, sigma_values,
                                n0, n, alpha, test_subset_size,
                                adversarial_examples, results_collector):
    """Perform randomized smoothing defense"""

    print("\n" + "="*60)
    print("RANDOMIZED SMOOTHING DEFENSE")
    print("="*60)


    #X_test_subset = X_test[:test_subset_size].astype(np.float32)
    #y_test_subset = y_test[:test_subset_size].astype(np.int64)


    X_test_subset = X_test.astype(np.float32)
    y_test_subset = y_test.astype(np.int64)


    for model_name, model in models.items():
        print(f"\n{model_name}")

        for sigma in sigma_values:
            defense_params = {
                'sigma': sigma,
                'n0': n0,
                'n': n,
                'alpha': alpha
            }

            print(f"  Sigma={sigma}")

            try:
                smoothed_clf = RandomizedSmoothing(
                    base_classifier=model,
                    sigma=sigma,
                    n0=n0,
                    n=n,
                    alpha=alpha
                )

                # Evaluate on clean data
                clean_results = smoothed_clf.certify(X_test_subset, y_test_subset)
                results_collector.add_defense_result(
                    model_name, 'Randomized_Smoothing', defense_params,
                    'Clean_Certified', {},
                    clean_results['accuracy'],
                    clean_results['precision'],
                    clean_results['recall'],
                    clean_results['f1score'],
                    clean_results['fpr'],
                    clean_results['fnr'],
                    clean_results['certified_accuracy'],
                    clean_results['mean_certified_radius']
                )

                # Evaluate on adversarial examples
                for adv_name, adv_data in adversarial_examples.items():
                    #adv_subset = adv_data['data'][:test_subset_size]
                    adv_subset = adv_data['data']
                    adv_results = smoothed_clf.certify(adv_subset, y_test_subset)

                    results_collector.add_defense_result(
                        model_name, 'Randomized_Smoothing', defense_params,
                        'Adversarial_Certified', {'attack': adv_name},
                        adv_results['accuracy'],
                        adv_results['precision'],
                        adv_results['recall'],
                        adv_results['f1score'],
                        adv_results['fpr'],
                        adv_results['fnr'],
                        adv_results['certified_accuracy'],
                        adv_results['mean_certified_radius']
                    )

            except Exception as e:
                print(f"    Error: {e}")

# ============================================================================
# RANDOMIZED SMOOTHING IMPLEMENTATION
# ============================================================================

class RandomizedSmoothing:
    """Randomized Smoothing Defense Implementation"""

    def __init__(self, base_classifier, sigma=0.25, n0=100, n=1000, alpha=0.001):
        self.base_classifier = base_classifier
        self.sigma = sigma
        self.n0 = n0
        self.n = n
        self.alpha = alpha
        self.rng = np.random.RandomState(RANDOM_SEED)

    def _add_noise(self, x, num_samples):
        if len(x.shape) == 1:
            x = x.reshape(1, -1)
        x = x.astype(np.float32)
        noise = self.rng.randn(num_samples, *x.shape).astype(np.float32) * self.sigma
        x_noisy = x + noise
        return x_noisy.reshape(-1, x.shape[1])


    def _get_predictions(self, x_noisy):
        try:
            if hasattr(self.base_classifier, 'predict'):
                if 'art' in str(type(self.base_classifier)).lower():
                    predictions = self.base_classifier.predict(x_noisy)
                    return np.argmax(predictions, axis=1)

            if hasattr(self.base_classifier, 'predict_proba'):
                probas = self.base_classifier.predict_proba(x_noisy)
                return np.argmax(probas, axis=1)

            return self.base_classifier.predict(x_noisy)

        except Exception as e:
            return self.base_classifier.predict(x_noisy)

    def predict(self, x):
        if len(x.shape) == 1:
            x = x.reshape(1, -1)

        x = x.astype(np.float32)
        batch_size = x.shape[0]
        all_predictions = []
        all_radii = []

        for i in range(batch_size):
            x_i = x[i:i+1]

            x_noisy_n0 = self._add_noise(x_i, self.n0)
            preds_n0 = self._get_predictions(x_noisy_n0)
            unique, counts = np.unique(preds_n0, return_counts=True)
            count_dict = dict(zip(unique, counts))
            top_class = max(count_dict, key=count_dict.get)

            x_noisy_n = self._add_noise(x_i, self.n)
            preds_n = self._get_predictions(x_noisy_n)
            top_count_n = np.sum(preds_n == top_class)
            p_a = top_count_n / self.n

            p_a = min(max(p_a, 1e-6), 1 - 1e-6)
            if p_a > 0.5:
                try:
                    from scipy import stats
                    radius = self.sigma * stats.norm.ppf(p_a)
                except:
                    radius = 0.0
            else:
                radius = 0.0

            all_predictions.append(top_class)
            all_radii.append(radius)

        return np.array(all_predictions), np.array(all_radii)

    def certify(self, x, true_labels):
        predictions, radii = self.predict(x)
        correct = (predictions == true_labels)
        certified = (radii > 0) & correct

        accuracy = accuracy_score(true_labels, predictions)
        precision = precision_score(true_labels, predictions, zero_division=0)
        recall = recall_score(true_labels, predictions, zero_division=0)
        f1 = f1_score(true_labels, predictions, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(true_labels, predictions).ravel()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0



        results = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1score': f1,
            'fpr': fpr,
            'fnr': fnr,
            'certified_accuracy': np.mean(certified),
            'mean_certified_radius': np.mean(radii[certified]) if np.sum(certified) > 0 else 0,
            'certification_rate': np.mean(certified)
        }

        return results

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    # Set all random seeds
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Set Python random seed
    import random
    random.seed(RANDOM_SEED)

    # Continue with existing code...
    create_output_directory()
    results_collector = ResultsCollector()

    print("="*60)
    print("ADVERSARIAL ROBUSTNESS EVALUATION FRAMEWORK")
    print("="*60)

    # Load and preprocess data
    print("\nLoading dataset...")
    data = pd.read_csv(DATASET_PATH)


    if(DATASET_NAME == 'CSECICIDS2018'):
      data['Label'] = data['Label'].apply(lambda x: 0 if x == 'Benign' else 1)
      exclude = ['Label']
      features = [col for col in data.columns if col not in exclude]
      print("Preprocessing and feature selection...")
      X, y, scaler, final_feats = preprocessing_and_feature_selection(data, label_col='Label', select_n=FEATURE_SELECTION)

    elif(DATASET_NAME == 'WUSTLEHMS2020'):
      columns_to_drop = ['Dir','Flgs','SrcAddr', 'DstAddr', 'Dport','SrcMac', 'DstMac', 'Packet_num', 'Attack Category']
      data = data.drop(columns=columns_to_drop, axis=1)

      # get a small sample of the dataset
      # Get unique classes
      classes = data['Label'].unique()

      # Calculate samples per class
      samples_per_class = 2000
      selected_samples = []

      # Select equal samples from each class
      for class_label in classes:
        class_data = data[data['Label'] == class_label]
        random_indices = np.random.choice(class_data.index, samples_per_class, replace=False)
        selected_samples.extend(random_indices)

      # Create final dataset with selected samples
      data = data.loc[selected_samples]

      exclude = ['Label']
      features = [col for col in data.columns if col not in exclude]
      print("Preprocessing and feature selection...")
      X, y, scaler, final_feats = preprocessing_and_feature_selection(data, label_col='Label', select_n=FEATURE_SELECTION)
    else: # TONIOT
      # remove columns with too many missing values
      df_processed = data.copy()
      # Replace '-' with NaN (missing values)
      df_processed = df_processed.replace('-', np.nan)

      # Calculate missing value percentage for each column
      missing_percentage = (df_processed.isnull().sum() / len(df_processed)) * 100

      # Identify columns with >50% missing values
      columns_to_drop = missing_percentage[missing_percentage > 50].index

      # Drop those columns
      df_processed = df_processed.drop(columns=columns_to_drop)

      # get a small sample of the dataset
      # Get unique classes
      classes = df_processed['label'].unique()

      # Calculate samples per class
      samples_per_class = 2000
      selected_samples = []

      # Select equal samples from each class
      for class_label in classes:
        class_data = df_processed[df_processed['label'] == class_label]
        random_indices = np.random.choice(class_data.index, samples_per_class, replace=False)
        selected_samples.extend(random_indices)

      # Create final dataset with selected samples
      df_processed = df_processed.loc[selected_samples]

      columns_to_drop = ['src_ip', 'dst_ip', 'type']
      df_processed = df_processed.drop(columns=columns_to_drop, axis=1)

      exclude = ['label']
      features = [col for col in df_processed.columns if col not in exclude]
      print("Preprocessing and feature selection...")
      X, y, scaler, final_feats = preprocessing_and_feature_selection(df_processed, label_col='label', select_n=FEATURE_SELECTION)


    print(f"Final feature set: {len(final_feats)} features")
    print(final_feats)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED
    )

    print(f"Training set: {len(X_train)} samples")
    print(f"Test set: {len(X_test)} samples")


    # Balance dataset
    if y_train.value_counts().min() / y_train.value_counts().max() < 0.75:
        smote = SMOTE(random_state=RANDOM_SEED)
        X_train, y_train = smote.fit_resample(X_train, y_train)

    # Train base models
    print("\n" + "="*60)
    print("TRAINING BASE MODELS")
    print("="*60)

    ml_models = {}
    for name, params in ML_MODELS_CONFIG.items():
        print(f"\nTraining {name}...")
        if name == 'RF':
            model = RandomForestClassifier(**params)
        elif name == 'SVM':
            model = LinearSVC(**params)
        else:
            model = LogisticRegression(**params)

        model.fit(X_train, y_train)
        ml_models[name] = model

        acc, prec, rec, f1, fpr, fnr = evaluate_model(model, X_test.values, y_test.values, 'sklearn')
        results_collector.add_base_result(name, acc, prec, rec, f1, fpr, fnr)
        print(f"  Accuracy: {acc:.4f}, F1-Score: {f1:.4f}")

    # Train MLP
    print("\nTraining MLP...")
    mlp = Net(
        X_train.shape[1],
        MLP_CONFIG['hidden_layers'],
        MLP_CONFIG['nb_classes']
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(mlp.parameters(), lr=MLP_CONFIG['lr'])

    classifier = PyTorchClassifier(
        model=mlp,
        loss=loss_fn,
        optimizer=optimizer,
        input_shape=(X_train.shape[1],),
        nb_classes=MLP_CONFIG['nb_classes'],
        clip_values=(np.min(X_train.values), np.max(X_train.values)),
        device_type='cpu'
    )

    classifier.fit(
        X_train.values.astype(np.float32),
        y_train.values.astype(np.int64),
        batch_size=MLP_CONFIG['batch_size'],
        nb_epochs=MLP_CONFIG['nb_epochs']
    )

    acc, prec, rec, f1, fpr, fnr = evaluate_model(
        classifier,
        X_test.values.astype(np.float32),
        y_test.values,
        'pytorch'
    )
    results_collector.add_base_result('MLP', acc, prec, rec, f1, fpr, fnr)
    print(f"  Accuracy: {acc:.4f}, F1-Score: {f1:.4f}")

    ml_models['MLP'] = classifier

    # Generate adversarial examples
    X_test_np = X_test.values.astype(np.float32)
    adversarial_examples = generate_attacks(classifier, X_test_np, ATTACK_CONFIGS)

    # Evaluate base models on adversarial examples
    print("\n" + "="*60)
    print("EVALUATING BASE MODELS ON ADVERSARIAL EXAMPLES")
    print("="*60)

    for adv_name, adv_data in adversarial_examples.items():
        print(f"\n{adv_name}:")
        attack_type = adv_name.split('_')[0]

        for model_name, model in ml_models.items():
            model_type = 'pytorch' if model_name == 'MLP' else 'sklearn'
            acc, prec, rec, f1, fpr, fnr, asr = evaluate_adversarial_attack(
                model,
                X_test.values,
                adv_data['data'],
                y_test.values,
                model_type
            )

            results_collector.add_attack_result(
                model_name, attack_type, adv_data['params'],
                acc, prec, rec, f1, fpr, fnr, asr
            )

            print(f"  {model_name}: Acc={acc:.4f}, ASR={asr:.4f}")

    # Adversarial Training Defense
    print("\n" + "="*60)
    print("ADVERSARIAL TRAINING DEFENSE")
    print("="*60)

    for attack_type in DEFENSE_CONFIGS['adversarial_training']['attack_types']:
        for ratio in DEFENSE_CONFIGS['adversarial_training']['ratio_values']:
            nb_epochs = DEFENSE_CONFIGS['adversarial_training']['nb_epochs']

            # Select a representative attack for training
            attack_key = [k for k in adversarial_examples.keys() if k.startswith(attack_type)][0]
            attack = adversarial_examples[attack_key]['attack']

            # Reset MLP for fair comparison
            mlp_fresh = Net(
                X_train.shape[1],
                MLP_CONFIG['hidden_layers'],
                MLP_CONFIG['nb_classes']
            )
            loss_fn = nn.CrossEntropyLoss()
            optimizer = optim.Adam(mlp_fresh.parameters(), lr=MLP_CONFIG['lr'])

            classifier_fresh = PyTorchClassifier(
                model=mlp_fresh,
                loss=loss_fn,
                optimizer=optimizer,
                input_shape=(X_train.shape[1],),
                nb_classes=MLP_CONFIG['nb_classes'],
                clip_values=(np.min(X_train.values), np.max(X_train.values)),
                device_type='cpu'
            )

            classifier_fresh.fit(
                X_train.values.astype(np.float32),
                y_train.values.astype(np.int64),
                batch_size=MLP_CONFIG['batch_size'],
                nb_epochs=MLP_CONFIG['nb_epochs']
            )

            adversarial_training_defense(
                classifier_fresh, X_train, y_train, X_test, y_test,
                attack, ratio, nb_epochs, adversarial_examples,
                {k: v for k, v in ml_models.items() if k != 'MLP'},
                results_collector
            )

    # Randomized Smoothing Defense
    randomized_smoothing_defense(
        ml_models,
        X_test.values,
        y_test.values,
        DEFENSE_CONFIGS['randomized_smoothing']['sigma_values'],
        DEFENSE_CONFIGS['randomized_smoothing']['n0'],
        DEFENSE_CONFIGS['randomized_smoothing']['n'],
        DEFENSE_CONFIGS['randomized_smoothing']['alpha'],
        DEFENSE_CONFIGS['randomized_smoothing']['test_subset_size'],
        adversarial_examples,
        results_collector
    )

    # Save all results
    print("\n" + "="*60)
    print("SAVING RESULTS")
    print("="*60)

    saved_files = results_collector.save_all_results()

    print(f"\nEvaluation complete!")
    print(f"Total files saved: {len(saved_files)}")

    return results_collector

# Run the framework
if __name__ == "__main__":
    results = main()

Results will be saved to: drive/My Drive/adversarial_results3
ADVERSARIAL ROBUSTNESS EVALUATION FRAMEWORK

Loading dataset...
Preprocessing and feature selection...
Finding optimal number of features...
Features: 5, Mean F1-Score: 0.9744
Features: 10, Mean F1-Score: 0.9895
Features: 15, Mean F1-Score: 0.9913
Features: 20, Mean F1-Score: 0.9930
Features: 25, Mean F1-Score: 0.9925
Features: 30, Mean F1-Score: 0.9895
Features: 35, Mean F1-Score: 0.9924
Features: 40, Mean F1-Score: 0.9919
Features: 45, Mean F1-Score: 0.9895
Features: 50, Mean F1-Score: 0.9919

Optimal number of features: 20 (F1-Score: 0.9930)
Final feature set: 20 features
Index(['Init Fwd Win Byts', 'Fwd Pkt Len Mean', 'Dst Port', 'Fwd Seg Size Avg',
       'TotLen Fwd Pkts', 'Subflow Fwd Byts', 'Bwd Pkts/s', 'Fwd IAT Min',
       'Flow Byts/s', 'Pkt Len Mean', 'Fwd IAT Tot', 'Flow IAT Std',
       'Flow IAT Max', 'Flow IAT Mean', 'Flow Duration', 'Fwd Header Len',
       'Fwd IAT Mean', 'Fwd Pkts/s', 'Subflow Bwd Pkts', 

PGD - Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  - PGD_eps0.15_iter16


PGD - Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  - PGD_eps0.15_iter25


...

Adversarial training epochs:   0%|          | 0/8 [00:00<?, ?it/s]

  MLP on clean: Acc=0.9084, F1=0.8643

Adversarial Training: PGD_eps0.15_iter10, ratio=0.5, epochs=8


Precompute adv samples:   0%|          | 0/1 [00:00<?, ?it/s]

Adversarial training epochs:   0%|          | 0/8 [00:00<?, ?it/s]

  MLP on clean: Acc=0.9349, F1=0.8996

RANDOMIZED SMOOTHING DEFENSE

RF
  Sigma=0.1
  Sigma=0.25
  Sigma=0.5
  Sigma=1.0

SVM
  Sigma=0.1
  Sigma=0.25
  Sigma=0.5
  Sigma=1.0

LR
  Sigma=0.1
  Sigma=0.25
  Sigma=0.5
  Sigma=1.0

MLP
  Sigma=0.1
  Sigma=0.25
  Sigma=0.5
  Sigma=1.0

SAVING RESULTS
Saved: drive/My Drive/adversarial_results3/base_results_CSECICIDS2018.csv
Saved: drive/My Drive/adversarial_results3/attack_results_CSECICIDS2018.csv
Saved: drive/My Drive/adversarial_results3/defense_results_CSECICIDS2018.csv

Evaluation complete!
Total files saved: 3
